# Notebook 03: KNN + MLP Hybrid Exploration

This notebook turns the early KNN / MLP branch into a real exploratory experiment. Its role is to test whether a classical instance-based model, a small neural network, and a simple probability blend can provide a meaningful alternative before the project later commits to boosting models.

## Motivation

The KNN + MLP idea was appealing early in the project for three reasons:
- KNN can capture local neighborhood structure without a strong parametric assumption.
- MLP can model nonlinear relationships that simple tabular baselines may miss.
- A hybrid perspective offers a useful contrast to later tree-based models.

In other words, this notebook is meant to be a genuine model-family exploration step rather than a tuning variation of the final approach.

## Conceptual Design

| Component | Intended strength | Practical limitation in this project |
| --- | --- | --- |
| KNN | Uses local similarity structure directly | Becomes expensive and less stable in a large, mixed-feature tabular dataset |
| MLP | Can learn nonlinear interactions automatically | Requires more tuning effort and training-time sensitivity than the project budget comfortably allowed |
| Hybrid view | Encourages complementary reasoning about local patterns and learned representations | Added complexity may not justify itself if the blended score remains close to the stronger single model |

The notebook therefore keeps the setup intentionally simple and leakage-safe.

## Experimental Setup

To keep the notebook executable in a local course-project workflow, the experiment uses:
- the shared project data-loading logic
- the shared engineered-feature helper
- leakage-safe preprocessing inside a single stratified holdout split
- a stratified sample cap by default for practical runtime

This notebook now uses one validation split rather than cross-validation. That keeps the experiment lightweight and easier to read as an early exploratory checkpoint.

If you want to stress-test the models more aggressively, you can raise the sample cap or set it to `None`, but the runtime will increase substantially.


In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import DATA_DIR, SUBMISSION_DIR, RANDOM_STATE, TARGET
from src.features import add_engineered_features
from src.train_xgb_cv import load_data

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

print(f'Project root: {PROJECT_ROOT}')
print(f'Data directory: {DATA_DIR}')


Project root: c:\Users\LENOVO\Desktop\machinelearning_project
Data directory: c:\Users\LENOVO\Desktop\machinelearning_project\data


In [2]:
# Runtime-oriented notebook settings
MAX_TRAIN_ROWS = 30000      # Set to None to use the full training set
VALID_SIZE = 0.20          # Single holdout ratio for this exploratory notebook
KNN_NEIGHBORS = 21
KNN_WEIGHTS = 'distance'
KNN_N_JOBS = 1
MLP_HIDDEN_LAYERS = (128, 64)
MLP_MAX_ITER = 80
HYBRID_WEIGHT_KNN = 0.5
HYBRID_WEIGHT_MLP = 0.5
SAVE_SUBMISSIONS = False    # Optional exploratory export; disabled by default

print('Notebook configuration:')
print(f'- MAX_TRAIN_ROWS = {MAX_TRAIN_ROWS}')
print(f'- VALID_SIZE = {VALID_SIZE}')
print(f'- KNN_NEIGHBORS = {KNN_NEIGHBORS}')
print(f'- MLP_HIDDEN_LAYERS = {MLP_HIDDEN_LAYERS}')
print(f'- KNN_N_JOBS = {KNN_N_JOBS}')
print(f'- SAVE_SUBMISSIONS = {SAVE_SUBMISSIONS}')


Notebook configuration:
- MAX_TRAIN_ROWS = 30000
- VALID_SIZE = 0.2
- KNN_NEIGHBORS = 21
- MLP_HIDDEN_LAYERS = (128, 64)
- KNN_N_JOBS = 1
- SAVE_SUBMISSIONS = False


## Data Loading and Feature Preparation

This section reuses the project's standard train/test loading function and the shared engineered-feature helper. The engineered features are row-wise combinations of existing columns, so they can be added before the train/validation split without introducing target leakage.

The notebook also drops `id` from the predictive feature set and optionally draws a stratified subsample for practical runtime.


In [3]:
train_df, test_df, sample_sub = load_data()

if MAX_TRAIN_ROWS is not None and len(train_df) > MAX_TRAIN_ROWS:
    sampled_idx, _ = train_test_split(
        train_df.index,
        train_size=MAX_TRAIN_ROWS,
        stratify=train_df[TARGET],
        random_state=RANDOM_STATE,
    )
    train_df = train_df.loc[sampled_idx].sort_values('id').reset_index(drop=True)
    print()
    print(f'Using a stratified training subset: {train_df.shape}')
else:
    print()
    print(f'Using the full training data: {train_df.shape}')

X = train_df.drop(columns=['id', TARGET]).copy()
y = train_df[TARGET].astype(int).copy()
X_test = test_df.drop(columns=['id']).copy()

X = add_engineered_features(X)
X_test = add_engineered_features(X_test)

categorical_cols = [
    col for col in [
        'gender', 'ethnicity', 'education_level',
        'income_level', 'smoking_status', 'employment_status'
    ]
    if col in X.columns
]
numeric_cols = [col for col in X.columns if col not in categorical_cols]

summary_df = pd.DataFrame({
    'item': ['train_rows', 'test_rows', 'feature_count', 'categorical_features', 'numeric_features', 'target_positive_rate'],
    'value': [len(train_df), len(test_df), X.shape[1], len(categorical_cols), len(numeric_cols), round(float(y.mean()), 4)],
})

display(summary_df)
X.head()


1. Loading data...
   Train: (700000, 26)
   Test: (300000, 25)

Using a stratified training subset: (30000, 26)


,item,value
0,train_rows,30000.0000
1,test_rows,300000.0000
2,feature_count,29.0000
3,categorical_features,6.0000
4,numeric_features,23.0000
5,target_positive_rate,0.6233


,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,diastolic_bp,heart_rate,cholesterol_total,hdl_cholesterol,ldl_cholesterol,triglycerides,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,age_family_history,age_bmi,cardio_risk_score,cholesterol_ratio,non_hdl_cholesterol
0,47,4,239,7.8,5.0,5.1,22.8,0.84,108,78,53,205,61,124,137,Female,White,Graduate,Middle,Current,Employed,0,0,0,0,1071.6,0,3.306452,144
1,41,2,47,5.5,7.6,5.3,23.9,0.84,101,77,70,155,56,77,124,Female,White,Highschool,Upper-Middle,Never,Employed,0,1,0,0,979.9,2,2.719298,99
2,60,3,48,6.1,5.4,9.8,25.2,0.85,100,78,76,161,46,81,97,Male,White,Graduate,Lower-Middle,Never,Employed,0,0,0,0,1512.0,0,3.425532,115
3,50,1,57,6.4,8.3,7.9,23.7,0.87,120,91,70,182,54,104,103,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,0,1185.0,0,3.309091,128
4,36,2,57,7.2,6.2,7.9,26.2,0.86,115,77,76,203,50,123,159,Female,White,Highschool,Middle,Never,Employed,0,1,0,0,943.2,2,3.980392,153


## Leakage-Safe Preprocessing

KNN and MLP both require preprocessing that is more sensitive than tree-based boosting. To keep the evaluation honest:
- numeric columns are imputed and scaled using only the training split
- categorical columns are imputed and one-hot encoded using only the training split
- the full preprocessing pipeline is wrapped inside each model pipeline

This means the scaler and encoder never see the validation split during fitting.


In [4]:
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def make_preprocessor():
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ])

    categorical_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', make_one_hot_encoder()),
    ])

    return ColumnTransformer([
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols),
    ])


def build_knn_pipeline():
    return Pipeline([
        ('preprocess', make_preprocessor()),
        ('model', KNeighborsClassifier(
            n_neighbors=KNN_NEIGHBORS,
            weights=KNN_WEIGHTS,
            n_jobs=KNN_N_JOBS,
        )),
    ])


def build_mlp_pipeline():
    return Pipeline([
        ('preprocess', make_preprocessor()),
        ('model', MLPClassifier(
            hidden_layer_sizes=MLP_HIDDEN_LAYERS,
            activation='relu',
            alpha=1e-4,
            learning_rate_init=1e-3,
            max_iter=MLP_MAX_ITER,
            early_stopping=True,
            validation_fraction=0.1,
            n_iter_no_change=10,
            random_state=RANDOM_STATE,
        )),
    ])


print('Preprocessing and model factories are ready.')


Preprocessing and model factories are ready.


## Holdout Evaluation

The notebook evaluates three model outputs under the same stratified validation split:
- KNN probability predictions
- MLP probability predictions
- a simple 50/50 hybrid probability blend

The hybrid is intentionally simple: it is not a tuned stacker, just an exploratory blend of two model families.


In [5]:
def evaluate_knn_mlp_hybrid_holdout(X, y):
    X = X.reset_index(drop=True)
    y = y.reset_index(drop=True)

    X_train, X_valid, y_train, y_valid = train_test_split(
        X,
        y,
        test_size=VALID_SIZE,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    knn_pipe = build_knn_pipeline()
    mlp_pipe = build_mlp_pipeline()

    knn_pipe.fit(X_train, y_train)
    mlp_pipe.fit(X_train, y_train)

    knn_pred = knn_pipe.predict_proba(X_valid)[:, 1]
    mlp_pred = mlp_pipe.predict_proba(X_valid)[:, 1]
    hybrid_pred = HYBRID_WEIGHT_KNN * knn_pred + HYBRID_WEIGHT_MLP * mlp_pred

    summary_rows = []
    prediction_map = {
        'KNN': knn_pred,
        'MLP': mlp_pred,
        'KNN + MLP Hybrid': hybrid_pred,
    }

    for model_name, pred in prediction_map.items():
        summary_rows.append({
            'model': model_name,
            'validation_auc': float(roc_auc_score(y_valid, pred)),
            'train_size': int(len(X_train)),
            'validation_size': int(len(X_valid)),
            'train_positive_rate': float(y_train.mean()),
            'validation_positive_rate': float(y_valid.mean()),
        })

    comparison_df = pd.DataFrame(summary_rows).sort_values('validation_auc', ascending=False).reset_index(drop=True)

    holdout_artifacts = {
        'train_indices': X_train.index.to_numpy(),
        'valid_indices': X_valid.index.to_numpy(),
        'y_valid': y_valid.reset_index(drop=True),
        'predictions': prediction_map,
    }

    fitted_full_models = {}
    if SAVE_SUBMISSIONS:
        print()
        print('Fitting full-data exploratory models for optional submissions...')
        fitted_full_models['KNN'] = build_knn_pipeline().fit(X, y)
        fitted_full_models['MLP'] = build_mlp_pipeline().fit(X, y)

    return comparison_df, holdout_artifacts, fitted_full_models


comparison_df, holdout_artifacts, fitted_full_models = evaluate_knn_mlp_hybrid_holdout(X, y)


## Results Comparison

The table below summarizes holdout ROC-AUC on the shared validation split. For this notebook, the most important reading is whether the hybrid provides a meaningful gain over its components and whether the non-boosting branch looks competitive enough to justify deeper investment.


In [6]:
if 'comparison_df' not in globals():
    raise RuntimeError('Run the holdout evaluation cell above first so that comparison_df is created.')

comparison_view = comparison_df.copy()
comparison_view['validation_auc'] = comparison_view['validation_auc'].map(lambda x: f'{x:.6f}')
comparison_view['train_positive_rate'] = comparison_view['train_positive_rate'].map(lambda x: f'{x:.6f}')
comparison_view['validation_positive_rate'] = comparison_view['validation_positive_rate'].map(lambda x: f'{x:.6f}')

display(comparison_view)

best_model_name = comparison_df.iloc[0]['model']
best_model_auc = comparison_df.iloc[0]['validation_auc']
second_model_auc = comparison_df.iloc[1]['validation_auc']

print(f'Best holdout AUC: {best_model_name} ({best_model_auc:.6f})')
print(f'Margin over the next model: {best_model_auc - second_model_auc:.6f}')


,model,validation_auc,train_size,validation_size,train_positive_rate,validation_positive_rate
0,MLP,0.689012,24000,6000,0.623292,0.623333
1,KNN + MLP Hybrid,0.676256,24000,6000,0.623292,0.623333
2,KNN,0.644678,24000,6000,0.623292,0.623333


Best holdout AUC: MLP (0.689012)
Margin over the next model: 0.012755


## Run-Level Interpretation

This cell turns the current holdout comparison table into a short narrative reading of the experiment run.


In [7]:
if 'comparison_df' not in globals():
    raise RuntimeError('Run the holdout evaluation cell above first so that comparison_df is created.')

best_row = comparison_df.iloc[0]
worst_row = comparison_df.iloc[-1]

if 'Hybrid' in best_row['model']:
    conclusion = 'The hybrid is the strongest option in this run, suggesting that the two model families provide complementary signal on the current holdout split.'
elif best_row['model'] == 'MLP':
    conclusion = 'MLP is the strongest model in this run, while the simple hybrid does not improve enough to justify replacing it.'
else:
    conclusion = 'KNN leads this run, which would be unusual for this project and worth investigating further.'

display(Markdown(
    f"""
**Current reading of the experiment**

- Best model by holdout AUC: **{best_row['model']}** ({best_row['validation_auc']:.6f})
- Lowest model by holdout AUC: **{worst_row['model']}** ({worst_row['validation_auc']:.6f})
- Gap between first and second place: **{comparison_df.iloc[0]['validation_auc'] - comparison_df.iloc[1]['validation_auc']:.6f}**
- Practical interpretation: {conclusion}

For this project, that makes the KNN / MLP branch informative as an exploration step, but still weaker and less scalable than the later boosting-centered workflow.
"""
))



**Current reading of the experiment**

- Best model by holdout AUC: **MLP** (0.689012)
- Lowest model by holdout AUC: **KNN** (0.644678)
- Gap between first and second place: **0.012755**
- Practical interpretation: MLP is the strongest model in this run, while the simple hybrid does not improve enough to justify replacing it.

For this project, that makes the KNN / MLP branch informative as an exploration step, but still weaker and less scalable than the later boosting-centered workflow.


## Optional Submission Export

This branch is exploratory, so Kaggle submissions are disabled by default. If you want to export notebook-level submission files for inspection, set `SAVE_SUBMISSIONS = True` and, ideally, set `MAX_TRAIN_ROWS = None` so the full training set is used.

In [8]:
if SAVE_SUBMISSIONS:
    knn_test_pred = fitted_full_models['KNN'].predict_proba(X_test)[:, 1]
    mlp_test_pred = fitted_full_models['MLP'].predict_proba(X_test)[:, 1]
    hybrid_test_pred = HYBRID_WEIGHT_KNN * knn_test_pred + HYBRID_WEIGHT_MLP * mlp_test_pred

    export_map = {
        '03_knn_submission.csv': knn_test_pred,
        '03_mlp_submission.csv': mlp_test_pred,
        '03_knn_mlp_hybrid_submission.csv': hybrid_test_pred,
    }

    for filename, pred in export_map.items():
        submission = sample_sub.copy()
        submission[TARGET] = pred
        out_path = SUBMISSION_DIR / filename
        submission.to_csv(out_path, index=False)
        print(f'Saved submission: {out_path}')
else:
    print('Submission export skipped. This notebook remains an exploratory comparison by default.')

Submission export skipped. This notebook remains an exploratory comparison by default.


## Short Takeaway

These results should be interpreted as an early model-family exploration rather than a final modeling decision.

A practical reading is:
- if one of the three variants is clearly ahead on the holdout split, that variant is the best representative of this branch
- if the hybrid only matches the best single model, then the added complexity is probably not worthwhile
- if all three results remain below the later boosting notebooks, this branch still serves as a useful comparison rather than a final direction

That is the main purpose of this notebook: to document the KNN / MLP branch clearly, then justify why the project later concentrates on stronger boosting-based methods.


## Connection to Later Notebooks

This notebook should be read as the final exploratory checkpoint before the project moves into the boosting-focused sequence:
- Notebook 04 establishes the default XGBoost baseline.
- Notebook 04.5 provides the default LightGBM comparison.
- Notebooks 05-09 then refine the boosted-model branch in a more systematic way.

A cautious conclusion is that the hybrid idea helps narrow the search space, even if it does not become the preferred final methodology.